# 08 · Base models play Kuhn Poker vs. "never fold"
SPIRAL's `KuhnPoker-v1` rules (our TextArena 0.7.4 port), run exactly as UnstableBaselines runs training games:
`qwen3-zs` prompt, action from the last `\boxed{}`, first invalid move loses, temperature 0.6, 4096 tokens.
200 matches per model, seats alternate. Reports win rate, invalid-move losses and format errors (no `\boxed{}`).
Roughly 15–40 minutes per model on a T4.

In [ ]:
# Setup (safe to re-run): GPU check, code, dependencies, benchmark data, output folder.
import os, pathlib, subprocess, sys
USE_DRIVE = False       # True: keep outputs on Google Drive so a new session resumes them
BRANCH = "main"
REPO = pathlib.Path("/content/cmpe260-selfplay")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > T4 GPU")
if not REPO.exists():
    from google.colab import userdata  # repository is private: add a read-only GH_TOKEN in Colab Secrets
    url = f"https://{userdata.get('GH_TOKEN')}@github.com/Anuragbo/selfplay.git"
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)

os.chdir(REPO)

def sh(cmd, log="setup.log"):
    """Run a shell command; stdout to this cell, stderr to a log file whose tail is shown if it fails."""
    with open(log, "a") as err:
        p = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=err, text=True)
    print(p.stdout[-20000:])
    if p.returncode:
        print(f"FAILED ({p.returncode}): {cmd}\n--- last lines of {log} ---")
        print("".join(open(log).readlines()[-60:]))
        raise RuntimeError(cmd)

print(sys.version)
if not pathlib.Path(".deps_ok").exists():
    sh("pip install -q textarena==0.7.4 'math-verify[antlr4_13_2]==0.9.0'")
    try:  # vLLM is optional: without it the transformers backend is used
        sh("pip install -q vllm", log="vllm-install.log")
    except RuntimeError:
        print("vLLM did not install; the smoke test will fall back to the transformers backend")
    pathlib.Path(".deps_ok").touch()
if not pathlib.Path("data/raw/aime25.jsonl").exists():
    sh("bash data/fetch_benchmarks.sh")
OUT = pathlib.Path("/content/outputs")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = pathlib.Path("/content/drive/MyDrive/CMPE260_selfplay/outputs")
OUT.mkdir(parents=True, exist_ok=True)
sh('python -c "import sys; sys.path.insert(0, \".\"); from selfplay.generation import versions; print(versions())"')
sys.path.insert(0, str(REPO))
print("outputs ->", OUT)

In [ ]:
# Backend chosen by 00_smoke (vllm, vllm with VLLM_USE_V1=0, or hf). Change it here if needed.
BACKEND = pathlib.Path("backend.txt").read_text().strip() if pathlib.Path("backend.txt").exists() else "vllm"
if BACKEND == "vllm-v0":
    os.environ["VLLM_USE_V1"] = "0"
BACKEND_FLAG = "hf" if BACKEND == "hf" else "vllm"
print("backend:", BACKEND)

In [ ]:
sh(f"python -m selfplay.play_kuhn --model Qwen/Qwen3-1.7B-Base --backend {BACKEND_FLAG} --opponents never_fold "
   f"--matches 200 --temperature 0.6 --max-tokens 4096 --run-id kuhn-base-qwen3-1.7b --out-dir {OUT}/kuhn-base-qwen3-1.7b "
   f"--print-summary", log="kuhn-base-qwen3-1.7b.log")

In [ ]:
sh(f"python -m selfplay.play_kuhn --model Qwen/Qwen3-4B-Base --backend {BACKEND_FLAG} --opponents never_fold "
   f"--matches 200 --temperature 0.6 --max-tokens 4096 --run-id kuhn-base-qwen3-4b --out-dir {OUT}/kuhn-base-qwen3-4b "
   f"--print-summary", log="kuhn-base-qwen3-4b.log")